In [12]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
import ipywidgets as widgets

In [13]:
def ellipse_points(mean, cov, n_std=2.0, num=300):
    """Generate points for an ellipse representing a Gaussian contour"""
    # eigen-decomposition for ellipse axes
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals = vals[order]
    vecs = vecs[:, order]

    t = np.linspace(0, 2*np.pi, num)
    circle = np.vstack([np.cos(t), np.sin(t)])  # (2, num)

    axes = n_std * np.sqrt(vals)                # radii along principal axes
    ell = vecs @ (axes[:, None] * circle)       # (2, num)
    ell = ell + mean[:, None]
    return ell

In [14]:
def plot_gaussians(a=9.0, b=2.0, c=1.0, u1=1.0, u2=0.0):
    """
    Plot two Gaussian distributions with covariance matrix [[a, b], [b, c]]
    and mean vector [u1, u2]
    
    Parameters:
    - a: top-left element of covariance matrix
    - b: off-diagonal element (correlation)
    - c: bottom-right element of covariance matrix
    - u1: first component of mean vector (x1 direction)
    - u2: second component of mean vector (x2 direction)
    """
    
    # Create covariance matrix
    Sigma = np.array([[a, b],
                      [b, c]])
    
    # Check if matrix is positive definite
    try:
        # Test if we can compute eigenvalues and they're positive
        eig_vals = np.linalg.eigvalsh(Sigma)
        if np.any(eig_vals <= 0):
            print(f"⚠️  Matrix not positive definite! Eigenvalues: {eig_vals}")
            print(f"   For positive definite: need a > 0, c > 0, and a*c > b^2")
            print(f"   Current: a={a:.2f}, c={c:.2f}, a*c={a*c:.2f}, b^2={b**2:.2f}")
            return
    except np.linalg.LinAlgError:
        print("⚠️  Invalid covariance matrix!")
        return
    
    # Mean vector (adjustable)
    mu = np.array([u1, u2])
    
    # Means for positive and negative classes
    m_pos = mu
    m_neg = -mu
    
    # Decision vector (Bayes/LDA)
    w = np.linalg.solve(Sigma, mu)  # = Sigma^{-1} mu
    
    # ---- Create Plot ----
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Ellipses (2-sigma contours)
    Epos = ellipse_points(m_pos, Sigma, n_std=2.0)
    Eneg = ellipse_points(m_neg, Sigma, n_std=2.0)
    ax.plot(Epos[0], Epos[1], 'b-', linewidth=2, label="+1 class (2σ)")
    ax.plot(Eneg[0], Eneg[1], 'r-', linewidth=2, label="-1 class (2σ)")
    
    # Sample points from the distributions
    rng = np.random.default_rng(42)  # Fixed seed for consistency
    Xpos = rng.multivariate_normal(m_pos, Sigma, size=200)
    Xneg = rng.multivariate_normal(m_neg, Sigma, size=200)
    ax.scatter(Xpos[:, 0], Xpos[:, 1], s=10, alpha=0.4, c='blue')
    ax.scatter(Xneg[:, 0], Xneg[:, 1], s=10, alpha=0.4, c='red')
    
    # Decision boundary w^T x = 0
    if abs(w[1]) > 1e-12:
        x1 = np.linspace(-8, 8, 200)
        x2 = -(w[0]/w[1]) * x1
        ax.plot(x1, x2, 'g--', linewidth=2, label="boundary w^T x = 0")
    else:
        ax.axvline(0.0, color='green', linestyle='--', linewidth=2, label="boundary x1 = 0")
    
    # Styling
    ax.set_xlim(-8, 8)
    ax.set_ylim(-5, 5)
    ax.set_aspect("equal", "box")
    ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
    ax.axvline(0, color='gray', linewidth=0.8, alpha=0.5)
    ax.grid(True, alpha=0.3)
    
    # Title with matrix info
    title = f"Two Gaussians with shared covariance + Bayes/LDA boundary\n"
    title += f"Mean μ = [{u1:.2f}, {u2:.2f}], Covariance Σ = [[{a:.2f}, {b:.2f}], [{b:.2f}, {c:.2f}]]"
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel("x1", fontsize=11)
    ax.set_ylabel("x2", fontsize=11)
    ax.legend(loc='upper right')
    
    # Display eigenvalues and determinant
    det = a * c - b**2
    info_text = f"det(Σ) = {det:.3f}\neigenvalues: [{eig_vals[0]:.3f}, {eig_vals[1]:.3f}]"
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()

print("✓ Functions defined!")

✓ Functions defined!


In [ ]:
# Create interactive plot with sliders
# Note: For positive definite matrix, need: a > 0, c > 0, and a*c > b^2

interact(plot_gaussians,
    a=FloatSlider(value=9.0, min=0.1, max=15.0, step=0.1, 
                  description='a (σ₁²):', continuous_update=False,
                  style={'description_width': '80px'}),
    b=FloatSlider(value=2.0, min=-10.0, max=10.0, step=0.1, 
                  description='b (cov):', continuous_update=False,
                  style={'description_width': '80px'}),
    c=FloatSlider(value=1.0, min=0.1, max=15.0, step=0.1, 
                  description='c (σ₂²):', continuous_update=False,
                  style={'description_width': '80px'}),
    u1=FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, 
                   description='u1 (μ₁):', continuous_update=False,
                   style={'description_width': '80px'}),
    u2=FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, 
                   description='u2 (μ₂):', continuous_update=False,
                   style={'description_width': '80px'})
);

interactive(children=(FloatSlider(value=9.0, continuous_update=False, description='a (σ₁²):', max=15.0, min=0.…

## Notes

### Mean Vector
The mean vector is:
```
μ = [u1, u2]
```

Where:
- **u1**: Mean in x₁ direction (horizontal center of positive class)
- **u2**: Mean in x₂ direction (vertical center of positive class)
- Positive class is centered at **+μ** = [u1, u2]
- Negative class is centered at **-μ** = [-u1, -u2]

### Covariance Matrix Structure
The covariance matrix is:
```
Σ = [[a, b],
     [b, c]]
```

Where:
- **a**: Variance in x₁ direction (controls horizontal spread)
- **c**: Variance in x₂ direction (controls vertical spread)
- **b**: Covariance (controls correlation/tilt of ellipses)

### Positive Definiteness Constraint
For a valid covariance matrix, it must be **positive definite**, which requires:
1. `a > 0` (positive variance)
2. `c > 0` (positive variance)
3. `a * c > b²` (determinant must be positive)

If these conditions aren't met, you'll see a warning message.

### What the Plot Shows
- **Blue ellipse/points**: Positive class centered at (+1, 0)
- **Red ellipse/points**: Negative class centered at (-1, 0)
- **Green dashed line**: Bayes optimal decision boundary
- The ellipses show 2σ confidence regions (≈95% of the probability mass)

### Try These Examples

**Covariance examples:**
1. **Circular (uncorrelated)**: a=4, b=0, c=4 → circles
2. **Horizontal stretch**: a=9, b=0, c=1 → horizontal ellipses
3. **Positive correlation**: a=9, b=2, c=1 → tilted ellipses (↗)
4. **Negative correlation**: a=9, b=-2, c=1 → tilted ellipses (↘)
5. **High correlation**: a=4, b=1.9, c=4 → very tilted (but keep b² < a*c!)

**Mean examples:**
1. **Horizontal separation**: u1=2, u2=0 → classes separated along x-axis
2. **Vertical separation**: u1=0, u2=2 → classes separated along y-axis
3. **Diagonal separation**: u1=1.5, u2=1.5 → classes separated diagonally
4. **No separation**: u1=0, u2=0 → both classes at origin (decision boundary through origin)
